# 02 - Data Cleaning and Preprocessing

## AI-Assisted Hospital Operations & Patient Analytics Dashboard

This notebook starts **Stage 3: Data Cleaning and Preprocessing**.

The goal is to prepare the raw HMIS dataset for SQL analysis, Python EDA, and Power BI dashboard development.

In this notebook, we will:

- load the raw CSV files
- convert date columns
- create patient age and age groups
- calculate length of stay
- create year and month fields
- validate billing amounts
- investigate missing `billing_detail.reference_id`
- save cleaned files into `data/processed/`


## 1. Import Libraries

We use beginner-friendly Python libraries:

- `pandas` for data analysis
- `numpy` for simple numeric logic
- `Path` for folder paths


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## 2. Set Project Paths

This path logic works whether the notebook is opened from the project root or from the `notebooks/` folder.

In [2]:
current_path = Path.cwd()

if (current_path / "data").exists():
    project_root = current_path
else:
    project_root = current_path.parent

raw_data_path = project_root / "data" / "raw" / "hospital_data"
processed_data_path = project_root / "data" / "processed"

processed_data_path.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data path:", raw_data_path)
print("Processed data path:", processed_data_path)

Project root: C:\Users\devip\Desktop\Healthcare Projects\Healthcare-Analytics-Project
Raw data path: C:\Users\devip\Desktop\Healthcare Projects\Healthcare-Analytics-Project\data\raw\hospital_data
Processed data path: C:\Users\devip\Desktop\Healthcare Projects\Healthcare-Analytics-Project\data\processed


## 3. Load Raw CSV Files

We load each CSV file into a dictionary called `tables`.

This keeps the code organized because each table can be accessed by name, such as `tables["admission"]`.

In [3]:
csv_files = sorted(raw_data_path.glob("*.csv"))

tables = {}

for file_path in csv_files:
    table_name = file_path.stem
    tables[table_name] = pd.read_csv(file_path)

print(f"Loaded {len(tables)} tables")
list(tables.keys())

Loaded 19 tables


['admission',
 'bed',
 'billing',
 'billing_detail',
 'department',
 'diagnostic_test',
 'disease',
 'doctor',
 'drug',
 'drug_inventory',
 'drug_manufacturer',
 'employee',
 'insurance_provider',
 'patient',
 'patient_diagnostic',
 'patient_insurance',
 'prescription',
 'staff_assignment',
 'ward']

## 4. Clean Text Columns

Before deeper cleaning, we remove accidental leading or trailing spaces from text columns.

Example: `Paid ` becomes `Paid`.

This helps avoid duplicate-looking categories in dashboards.

In [4]:
for table_name, df in tables.items():
    text_columns = df.select_dtypes(include="object").columns
    for column in text_columns:
        df[column] = df[column].astype("string").str.strip()

print("Text columns stripped successfully.")

Text columns stripped successfully.


## 5. Convert Date Columns

Date columns often load as text when reading CSV files.

We convert them into proper datetime columns so we can calculate age, length of stay, and monthly trends.

In [5]:
date_columns = {
    "patient": ["date_of_birth"],
    "admission": ["admission_date", "discharge_date"],
    "billing": ["bill_date"],
    "patient_diagnostic": ["test_date"],
    "drug_inventory": ["last_restock_date"],
    "patient_insurance": ["policy_start_date", "policy_end_date"],
    "employee": ["date_of_joining"],
}

date_conversion_summary = []

for table_name, columns in date_columns.items():
    for column in columns:
        before_missing = tables[table_name][column].isna().sum()
        tables[table_name][column] = pd.to_datetime(tables[table_name][column], errors="coerce")
        after_missing = tables[table_name][column].isna().sum()
        date_conversion_summary.append({
            "table_name": table_name,
            "column_name": column,
            "missing_before_conversion": int(before_missing),
            "missing_after_conversion": int(after_missing),
            "new_missing_from_conversion": int(after_missing - before_missing),
        })

pd.DataFrame(date_conversion_summary)

,table_name,column_name,missing_before_conversion,missing_after_conversion,new_missing_from_conversion
0,patient,date_of_birth,0,0,0
1,admission,admission_date,0,0,0
2,admission,discharge_date,0,0,0
3,billing,bill_date,0,0,0
4,patient_diagnostic,test_date,0,0,0
5,drug_inventory,last_restock_date,0,0,0
6,patient_insurance,policy_start_date,0,0,0
7,patient_insurance,policy_end_date,0,0,0
8,employee,date_of_joining,0,0,0


## 6. Create Patient Age and Age Group

The raw dataset has `date_of_birth`, but dashboards usually need age groups.

For this project, we calculate age using the latest admission/discharge date in the dataset as the reference date. This keeps the age calculation aligned with the dataset period instead of the current real-world date.

In [6]:
dataset_reference_date = max(
    tables["admission"]["admission_date"].max(),
    tables["admission"]["discharge_date"].max()
)

patients_clean = tables["patient"].copy()

patients_clean["patient_age"] = (
    (dataset_reference_date - patients_clean["date_of_birth"]).dt.days / 365.25
).astype("int64")

age_bins = [-1, 12, 19, 34, 54, 69, np.inf]
age_labels = ["Child", "Teen", "Young Adult", "Adult", "Senior", "Elderly"]

patients_clean["age_group"] = pd.cut(
    patients_clean["patient_age"],
    bins=age_bins,
    labels=age_labels
)

print("Dataset reference date:", dataset_reference_date.date())
patients_clean[["patient_id", "date_of_birth", "patient_age", "age_group"]].head()

Dataset reference date: 2026-01-12


,patient_id,date_of_birth,patient_age,age_group
0,1,1987-08-24,38,Adult
1,2,1960-05-18,65,Senior
2,3,1955-04-24,70,Elderly
3,4,2004-06-16,21,Young Adult
4,5,1977-07-22,48,Adult


## 7. Create Admission Time Fields and Length of Stay

Length of stay is one of the most important hospital operations KPIs.

It is calculated as:

```text
discharge_date - admission_date
```

In [7]:
admissions_clean = tables["admission"].copy()

admissions_clean["length_of_stay_days"] = (
    admissions_clean["discharge_date"] - admissions_clean["admission_date"]
).dt.days

admissions_clean["admission_year"] = admissions_clean["admission_date"].dt.year
admissions_clean["admission_month"] = admissions_clean["admission_date"].dt.month
admissions_clean["admission_month_name"] = admissions_clean["admission_date"].dt.month_name()
admissions_clean["admission_year_month"] = admissions_clean["admission_date"].dt.to_period("M").astype(str)

admissions_clean[["admission_id", "admission_date", "discharge_date", "length_of_stay_days", "admission_year_month"]].head()

,admission_id,admission_date,discharge_date,length_of_stay_days,admission_year_month
0,1,2020-02-25,2020-02-27,2,2020-02
1,2,2022-02-22,2022-03-04,10,2022-02
2,3,2021-02-03,2021-02-09,6,2021-02
3,4,2021-12-31,2022-01-05,5,2021-12
4,5,2022-07-02,2022-07-07,5,2022-07


## 8. Check Length of Stay Quality

We check whether any admissions have negative length of stay. A negative value would mean the discharge date is before the admission date.

In [8]:
los_quality = {
    "total_admissions": len(admissions_clean),
    "negative_length_of_stay": int((admissions_clean["length_of_stay_days"] < 0).sum()),
    "zero_day_stays": int((admissions_clean["length_of_stay_days"] == 0).sum()),
    "average_length_of_stay": round(admissions_clean["length_of_stay_days"].mean(), 2),
    "maximum_length_of_stay": int(admissions_clean["length_of_stay_days"].max()),
}

pd.DataFrame([los_quality])

,total_admissions,negative_length_of_stay,zero_day_stays,average_length_of_stay,maximum_length_of_stay
0,45000,0,0,5.16,15


## 9. Create Billing Time Fields and Insurance Percentage

For the financial dashboard, we need monthly revenue trends and insurance coverage percentage.

In [9]:
billing_clean = tables["billing"].copy()

billing_clean["bill_year"] = billing_clean["bill_date"].dt.year
billing_clean["bill_month"] = billing_clean["bill_date"].dt.month
billing_clean["bill_month_name"] = billing_clean["bill_date"].dt.month_name()
billing_clean["bill_year_month"] = billing_clean["bill_date"].dt.to_period("M").astype(str)

billing_clean["insurance_coverage_percent"] = np.where(
    billing_clean["total_amount"] > 0,
    (billing_clean["insurance_covered_amount"] / billing_clean["total_amount"]) * 100,
    np.nan
)

billing_clean["insurance_coverage_percent"] = billing_clean["insurance_coverage_percent"].round(2)

billing_clean[["bill_id", "bill_date", "total_amount", "insurance_covered_amount", "patient_payable_amount", "insurance_coverage_percent"]].head()

,bill_id,bill_date,total_amount,insurance_covered_amount,patient_payable_amount,insurance_coverage_percent
0,1,2025-05-26,68483,61634.7,6848.3,90.0
1,2,2023-11-19,70917,35458.5,35458.5,50.0
2,3,2023-02-20,28137,25323.3,2813.7,90.0
3,4,2021-09-01,80665,64532.0,16133.0,80.0
4,5,2021-07-13,54920,27460.0,27460.0,50.0


## 10. Validate Billing Amounts

A useful financial quality check is whether:

```text
insurance_covered_amount + patient_payable_amount = total_amount
```

In [10]:
billing_clean["billing_amount_matches"] = np.isclose(
    billing_clean["insurance_covered_amount"] + billing_clean["patient_payable_amount"],
    billing_clean["total_amount"],
    atol=0.01
)

billing_quality = {
    "total_bills": len(billing_clean),
    "negative_total_amount": int((billing_clean["total_amount"] < 0).sum()),
    "negative_insurance_covered_amount": int((billing_clean["insurance_covered_amount"] < 0).sum()),
    "negative_patient_payable_amount": int((billing_clean["patient_payable_amount"] < 0).sum()),
    "insurance_greater_than_total": int((billing_clean["insurance_covered_amount"] > billing_clean["total_amount"]).sum()),
    "billing_amount_mismatch": int((~billing_clean["billing_amount_matches"]).sum()),
}

pd.DataFrame([billing_quality])

,total_bills,negative_total_amount,negative_insurance_covered_amount,negative_patient_payable_amount,insurance_greater_than_total,billing_amount_mismatch
0,45000,0,0,0,0,0


## 11. Investigate Missing `billing_detail.reference_id`

Stage 2 found many missing values in `billing_detail.reference_id`.

Here we check whether missing values are linked to specific charge types.

In [11]:
billing_detail_clean = tables["billing_detail"].copy()

billing_detail_clean["reference_id_missing"] = billing_detail_clean["reference_id"].isna()

reference_missing_by_charge_type = (
    billing_detail_clean
    .groupby("charge_type")
    .agg(
        total_records=("billing_detail_id", "count"),
        missing_reference_id=("reference_id_missing", "sum"),
        total_amount=("amount", "sum")
    )
    .reset_index()
)

reference_missing_by_charge_type["missing_reference_percent"] = (
    reference_missing_by_charge_type["missing_reference_id"] / reference_missing_by_charge_type["total_records"] * 100
).round(2)

reference_missing_by_charge_type.sort_values("missing_reference_percent", ascending=False)

,charge_type,total_records,missing_reference_id,total_amount,missing_reference_percent
0,Drug,31344,31344,50066094,100.0
1,Procedure,13575,13575,204517073,100.0
3,Test,22483,22483,61725515,100.0
2,Room,45000,0,1162571584,0.0


## 12. Check Important Categorical Values

This helps us confirm dashboard filters will be clean and readable.

In [12]:
categorical_checks = {
    "patient.gender": patients_clean["gender"].dropna().sort_values().unique(),
    "admission.admission_type": admissions_clean["admission_type"].dropna().sort_values().unique(),
    "admission.admission_status": admissions_clean["admission_status"].dropna().sort_values().unique(),
    "billing.payment_status": billing_clean["payment_status"].dropna().sort_values().unique(),
    "billing.payment_mode": billing_clean["payment_mode"].dropna().sort_values().unique(),
    "bed.bed_status": tables["bed"]["bed_status"].dropna().sort_values().unique(),
}

for column_name, values in categorical_checks.items():
    print(column_name)
    print(values)
    print("-")

patient.gender
<StringArray>
['Female', 'Male', 'Other']
Length: 3, dtype: string
-
admission.admission_type
<StringArray>
['Elective', 'Emergency']
Length: 2, dtype: string
-
admission.admission_status
<StringArray>
['Discharged']
Length: 1, dtype: string
-
billing.payment_status
<StringArray>
['Paid', 'Pending']
Length: 2, dtype: string
-
billing.payment_mode
<StringArray>
['Card', 'Cash', 'Insurance', 'UPI']
Length: 4, dtype: string
-
bed.bed_status
<StringArray>
['Available', 'Occupied']
Length: 2, dtype: string
-


## 13. Save Processed Files

We now save cleaned/processed files into `data/processed/`.

Important: raw files are not changed.

In [13]:
processed_tables = tables.copy()

processed_tables["patient"] = patients_clean
processed_tables["admission"] = admissions_clean
processed_tables["billing"] = billing_clean
processed_tables["billing_detail"] = billing_detail_clean

for table_name, df in processed_tables.items():
    output_path = processed_data_path / f"processed_{table_name}.csv"
    df.to_csv(output_path, index=False)

print(f"Saved {len(processed_tables)} processed files to:")
print(processed_data_path)

Saved 19 processed files to:
C:\Users\devip\Desktop\Healthcare Projects\Healthcare-Analytics-Project\data\processed


## 14. Stage 3 Summary

After running this notebook, update this section with your findings.

Expected outputs:

- cleaned patient table with age and age group
- cleaned admission table with length of stay and admission time fields
- cleaned billing table with billing time fields and insurance coverage percentage
- billing detail table with reference ID missing flag
- processed CSV files saved in `data/processed/`

Next stage after this notebook:

```text
Stage 4 - SQL Analytics and KPI Analysis
```